# TF-IDF + SVM

**TF-IDF** (Term Frequency-Inverse Document Frequency) converts each document into a very long, sparse vector of numbers. Each position in the vector corresponds to a word (or word pair) from vocabulary. The number at that position is high if the word appears often in this document but rarely in other documents in the corpus. Words that appear everywhere ("the", "and", "I") get low weights; words that are distinctive to certain documents get high weights.
For cognitive distortion classification, this is a surprisingly strong signal because distortions often have characteristic vocabulary: "always", "never", "everyone" for overgeneralization; "should", "must", "ought" for should statements; "failure", "worthless" for labelling.

**SVM** (Support Vector Machine) is a classifier that, given these vectors, finds hyperplanes separating the categories. For multi-class problems, scikit-learn implements one-vs-rest internally. SVM handles high-dimensional sparse features well, which is exactly what TF-IDF produces.

- The `stratify` argument in `train_test_split` is important because the classes are imbalanced -> without stratification, some rare classes might end up only in the test set. The `class_weight='balanced'` argument tells SVM to weight minority classes more heavily during training, which partially compensates for the class imbalance. Without it, the model just learns to predict the majority class.

- The n-gram range of (1, 2) — unigrams and bigrams — matches what Shreevastava and Foltz used. Going up to trigrams sometimes helps but often doesn't, and adds parameters.

- `stop_words='english'` is debatable. Removing stopwords helps for topic classification but can hurt for cognitive distortions because words like "should", "must", "always", "never" are on many stopword lists but are exactly the signal needed. Both are tested.

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import pandas as pd

# Load preprocessed TherapistQA from Stage 0
df = pd.read_csv('processed/therapistqa.csv')

# Split - use the same split every time for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    df['patient_question'],
    df['dominant_distortion'],
    test_size=0.2,
    stratify=df['dominant_distortion'],
    random_state=42
)

# Build pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),            # unigrams and bigrams
        min_df=2,                      # ignore words appearing in <2 documents
        max_df=0.95,                   # ignore words appearing in >95% of documents
        sublinear_tf=True,             # dampen very frequent terms
        stop_words='english'           # remove English stopwords
    )),
    ('svm', LinearSVC(
        C=1.0,                         # regularisation strength
        class_weight='balanced',       # important given class imbalance
        max_iter=5000
    ))
])

# Train
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

                         precision    recall  f1-score   support

All-or-nothing thinking      0.053     0.050     0.051        20
    Emotional Reasoning      0.188     0.111     0.140        27
        Fortune-telling      0.174     0.138     0.154        29
               Labeling      0.160     0.121     0.138        33
          Magnification      0.150     0.154     0.152        39
          Mental filter      0.130     0.125     0.128        24
           Mind Reading      0.321     0.354     0.337        48
          No Distortion      0.572     0.722     0.638       187
     Overgeneralization      0.114     0.083     0.096        48
        Personalization      0.067     0.033     0.044        30
      Should statements      0.238     0.238     0.238        21

               accuracy                          0.362       506
              macro avg      0.197     0.194     0.192       506
           weighted avg      0.317     0.362     0.335       506



### Binary detection number
The 11-way problem is reduced to 'distorted vs non-distorted'

In [2]:
import numpy as np
from sklearn.metrics import classification_report

# Collapse to binary
y_test_binary = np.where(y_test == 'No Distortion', 'No Distortion', 'Distorted')
y_pred_binary = np.where(y_pred == 'No Distortion', 'No Distortion', 'Distorted')

print(classification_report(y_test_binary, y_pred_binary, digits=3))

               precision    recall  f1-score   support

    Distorted      0.807     0.683     0.740       319
No Distortion      0.572     0.722     0.638       187

     accuracy                          0.698       506
    macro avg      0.690     0.703     0.689       506
 weighted avg      0.720     0.698     0.703       506



### Ablation: stopword removal
Keep stopwords.

In [3]:
# Build pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),            # unigrams and bigrams
        min_df=2,                      # ignore words appearing in <2 documents
        max_df=0.95,                   # ignore words appearing in >95% of documents
        sublinear_tf=True,             # dampen very frequent terms
    )),
    ('svm', LinearSVC(
        C=1.0,                         # regularisation strength
        class_weight='balanced',       # important given class imbalance
        max_iter=5000
    ))
])

# Train
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

                         precision    recall  f1-score   support

All-or-nothing thinking      0.077     0.050     0.061        20
    Emotional Reasoning      0.059     0.037     0.045        27
        Fortune-telling      0.240     0.207     0.222        29
               Labeling      0.056     0.030     0.039        33
          Magnification      0.100     0.077     0.087        39
          Mental filter      0.091     0.083     0.087        24
           Mind Reading      0.320     0.333     0.327        48
          No Distortion      0.560     0.775     0.650       187
     Overgeneralization      0.111     0.083     0.095        48
        Personalization      0.100     0.067     0.080        30
      Should statements      0.250     0.190     0.216        21

               accuracy                          0.366       506
              macro avg      0.178     0.176     0.174       506
           weighted avg      0.300     0.366     0.325       506



### Ablation: n-gram range
Use trigrams with unigrams

In [5]:
# Build pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 3),            # unigrams and trigrams
        min_df=2,                      # ignore words appearing in <2 documents
        max_df=0.95,                   # ignore words appearing in >95% of documents
        sublinear_tf=True,             # dampen very frequent terms
        stop_words='english'           # remove English stopwords
    )),
    ('svm', LinearSVC(
        C=1.0,                         # regularisation strength
        class_weight='balanced',       # important given class imbalance
        max_iter=5000
    ))
])

# Train
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

                         precision    recall  f1-score   support

All-or-nothing thinking      0.056     0.050     0.053        20
    Emotional Reasoning      0.158     0.111     0.130        27
        Fortune-telling      0.167     0.138     0.151        29
               Labeling      0.192     0.152     0.169        33
          Magnification      0.158     0.154     0.156        39
          Mental filter      0.160     0.167     0.163        24
           Mind Reading      0.320     0.333     0.327        48
          No Distortion      0.576     0.711     0.636       187
     Overgeneralization      0.111     0.083     0.095        48
        Personalization      0.000     0.000     0.000        30
      Should statements      0.217     0.238     0.227        21

               accuracy                          0.358       506
              macro avg      0.192     0.194     0.192       506
           weighted avg      0.315     0.358     0.333       506

